# 🔍 Preflight Check

This notebook checks the required environment components before starting RHOAI 3.5 τ-Knowledge model training.

## Checklist

| # | Item | Description |
|---|------|------|
| 1 | Python & GPU | Check Python version, CUDA availability, VRAM |
| 2 | Environment Variables | Load `.env` file and verify required variables |
| 3 | MLflow | Check MLflow tracking server connection |
| 4 | Base Model | `Qwen/Qwen3-4B-Instruct-2507` tokenizer load test |

All items ✅ means training can proceed.  
Fix the settings for any ❌ items first.

In [ ]:
"""Check Python version, GPU availability, and VRAM."""

import sys
import platform

results = {}

# Python version
py_version = sys.version_info
py_ok = py_version >= (3, 11)
results["python"] = py_ok
print(f"Python version: {sys.version}")
print(f"  Python >= 3.11: {'✅' if py_ok else '❌'}")
print(f"  Platform: {platform.platform()}")
print()

# GPU / CUDA check
try:
    import torch

    cuda_available = torch.cuda.is_available()
    results["cuda"] = cuda_available
    print(f"PyTorch version: {torch.__version__}")
    print(f"  CUDA available: {'✅' if cuda_available else '❌'}")

    if cuda_available:
        gpu_count = torch.cuda.device_count()
        print(f"  GPU count: {gpu_count}")
        for i in range(gpu_count):
            name = torch.cuda.get_device_name(i)
            vram_total = torch.cuda.get_device_properties(i).total_mem / (1024**3)
            vram_free = (torch.cuda.get_device_properties(i).total_mem - torch.cuda.memory_allocated(i)) / (1024**3)
            print(f"  GPU {i}: {name}")
            print(f"    Total VRAM: {vram_total:.1f} GB")
            print(f"    Available VRAM: {vram_free:.1f} GB")
            results["gpu_name"] = name
            results["vram_gb"] = round(vram_total, 1)
    else:
        print("  ⚠️  Data preparation and validation can proceed without GPU.")
        print("  ⚠️  GPU is required for LoRA/OSFT training.")
except ImportError:
    results["cuda"] = False
    print("❌ PyTorch is not installed.")
    print("  Run pip install torch.")

In [ ]:
"""Check .env loaded and required environment variables present."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import load_env, PROJECT_ROOT

# Load .env
env_path = PROJECT_ROOT / ".env"
env_exists = env_path.exists()
results["env_file"] = env_exists
print(f".env file exists: {'✅' if env_exists else '❌'} ({env_path})")

if env_exists:
    load_env(env_path)
else:
    print("  ⚠️  Copy .env.example to .env and fill in values.")
    print(f"  cp {PROJECT_ROOT / '.env.example'} {env_path}")

# Required variables for learner training path
required_vars = {
    "BASE_MODEL_ID": "Base model ID",
    "TOKENIZER_ID": "Tokenizer ID",
}

# Optional but recommended
optional_vars = {
    "MLFLOW_TRACKING_URI": "MLflow tracking URI",
    "S3_ENDPOINT": "S3 endpoint",
    "S3_BUCKET": "S3 bucket",
    "BASE_SERVING_ENDPOINT": "Base model serving endpoint",
}

print("\n--- Required environment variables ---")
all_required_ok = True
for var, desc in required_vars.items():
    val = os.environ.get(var, "")
    ok = bool(val)
    if not ok:
        all_required_ok = False
    status = "✅" if ok else "❌"
    display_val = val[:40] + "..." if len(val) > 40 else val
    print(f"  {status} {var} ({desc}): {display_val or '(not set)'}")

results["env_vars"] = all_required_ok

print("\n--- Optional environment variables ---")
for var, desc in optional_vars.items():
    val = os.environ.get(var, "")
    status = "✅" if val else "⚠️"
    display_val = val[:40] + "..." if len(val) > 40 else val
    print(f"  {status} {var} ({desc}): {display_val or '(not set)'}")

In [ ]:
"""Check working directories exist and are writable."""

work_dirs = {
    "data": PROJECT_ROOT / "data",
    "models": PROJECT_ROOT / "models",
    "checkpoints": PROJECT_ROOT / "checkpoints",
}

print("--- Working Directory Check ---")
all_dirs_ok = True
for name, p in work_dirs.items():
    exists = p.exists()
    writable = os.access(p, os.W_OK) if exists else False

    if exists and writable:
        status = "✅"
    elif exists:
        status = "⚠️  (read-only)"
        all_dirs_ok = False
    else:
        try:
            p.mkdir(parents=True, exist_ok=True)
            status = "✅ (newly created)"
        except OSError as exc:
            status = f"❌ (creation failed: {exc})"
            all_dirs_ok = False

    print(f"  {status} {name}: {p}")

results["work_dirs"] = all_dirs_ok


In [ ]:
"""Check MLflow tracking server connectivity."""

mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
mlflow_ok = False

print("--- MLflow Connection Check ---")

if not mlflow_uri:
    print("  ⚠️  MLFLOW_TRACKING_URI is not set.")
    print("  ⚠️  Training is possible but experiment tracking will not be recorded.")
    print("  ⚠️  Proceeding in --no-mlflow mode.")
else:
    print(f"  MLflow URI: {mlflow_uri}")
    try:
        import mlflow

        mlflow.set_tracking_uri(mlflow_uri)
        # Test connectivity by listing experiments
        experiments = mlflow.search_experiments(max_results=1)
        mlflow_ok = True
        print(f"  ✅ MLflow server connection successful (experiments: {len(experiments)}+)")

        # Check expected experiment names
        expected_experiments = [
            os.environ.get("MLFLOW_EXPERIMENT_DATA", "rhoai-model-training-lab-data"),
            os.environ.get("MLFLOW_EXPERIMENT_TRAINING", "rhoai-model-training-lab-training"),
            os.environ.get("MLFLOW_EXPERIMENT_EVAL", "rhoai-model-training-lab-evaluation"),
        ]
        for exp_name in expected_experiments:
            exp = mlflow.get_experiment_by_name(exp_name)
            if exp:
                print(f"    ✅ Experiment '{exp_name}' exists")
            else:
                print(f"    ⚠️  Experiment '{exp_name}' not created (will be auto-created during training)")
    except ImportError:
        print("  ❌ mlflow package is not installed.")
        print("  Run pip install mlflow.")
    except Exception as exc:
        print(f"  ❌ MLflow connection failed: {exc}")

results["mlflow"] = mlflow_ok

In [ ]:
"""Check base model accessibility — tokenizer load test."""

model_id = os.environ.get("BASE_MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507")
model_ok = False

print("--- Base Model Accessibility Check ---")
print(f"  Model ID: {model_id}")

try:
    from transformers import AutoTokenizer

    print("  Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model_ok = True

    print(f"  ✅ Tokenizer loaded successfully")
    print(f"    Vocabulary size: {tokenizer.vocab_size:,}")
    print(f"    Model max length: {getattr(tokenizer, 'model_max_length', 'N/A')}")

    # Chat template check
    has_chat_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template
    print(f"    Chat template: {'✅ present' if has_chat_template else '⚠️  absent'}")

    # Quick encode/decode test
    test_text = "Hello, I'd like to check my account balance."
    tokens = tokenizer.encode(test_text)
    decoded = tokenizer.decode(tokens)
    print(f"    Encode/decode test: ✅ ({len(tokens)} tokens)")

    # Tool call template test
    test_messages = [
        {"role": "system", "content": "You are a banking assistant."},
        {"role": "user", "content": "What is my balance?"},
    ]
    try:
        formatted = tokenizer.apply_chat_template(test_messages, tokenize=False)
        print(f"    Chat template applied: ✅ ({len(formatted)} chars)")
    except Exception as tmpl_err:
        print(f"    Chat template applied: ⚠️  {tmpl_err}")

except ImportError:
    print("  ❌ transformers package is not installed.")
except Exception as exc:
    print(f"  ❌ Tokenizer load failed: {exc}")
    print("  Network access to the model is required, or check local cache.")

results["model"] = model_ok

In [ ]:
"""Print summary with pass/fail indicators."""

from rich.console import Console
from rich.table import Table

console = Console()

table = Table(title="🔍 Preflight Check Results", show_header=True)
table.add_column("Item", style="bold")
table.add_column("Status")
table.add_column("Notes")

checks = [
    ("Python >= 3.11", results.get("python", False), f"v{sys.version_info.major}.{sys.version_info.minor}"),
    ("CUDA / GPU", results.get("cuda", False), results.get("gpu_name", "No GPU")),
    (".env file", results.get("env_file", False), ""),
    ("Required env vars", results.get("env_vars", False), ""),
    ("Working dirs", results.get("work_dirs", False), ""),
    ("MLflow", results.get("mlflow", False), mlflow_uri or "Not set"),
    ("Base model", results.get("model", False), model_id),
]

all_pass = True
critical_fail = False
for name, ok, note in checks:
    status = "✅ Pass" if ok else "❌ Fail"
    style = "green" if ok else "red"
    table.add_row(name, f"[{style}]{status}[/{style}]", str(note))
    if not ok:
        all_pass = False
        if name in ("Python >= 3.11", "Required env vars"):
            critical_fail = True

console.print(table)
print()

if all_pass:
    print("🎉 All preflight checks passed!")
    print("   Next step: Run 01_load_prepared_dataset.ipynb.")
elif critical_fail:
    print("🚫 Critical items failed. Follow the instructions above to fix them.")
else:
    print("⚠️  Some items failed but basic training may still be possible.")
    print("   Without GPU, only data validation can proceed.")